# Download Amenity Data for PropertyLens Map

Downloads the latest amenity location data from Singapore government sources (data.gov.sg and OneMap) and saves normalised CSVs to `data/amenities/`.

**Outputs:**
- `data/amenities/mrt_stations.csv` — MRT/LRT stations (deduplicated per station)
- `data/amenities/hawker_centres.csv` — Hawker centres
- `data/amenities/schools.csv` — Primary schools with geocoded lat/lng
- `data/amenities/malls.csv` — Shopping malls via OneMap POI search

**Data sources:** All free, no API key required.
- LTA MRT Station Exit GeoJSON via data.gov.sg
- NEA Hawker Centres GeoJSON via data.gov.sg
- MOE School Directory via data.gov.sg + OneMap geocoding
- OneMap POI search for shopping malls

In [ ]:
import requests
import pandas as pd
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent
OUT_DIR = REPO_ROOT / "data" / "amenities"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUT_DIR}")

## 1. MRT / LRT Stations

Source: LTA MRT Station Exit GeoJSON from data.gov.sg. We deduplicate to one point per station name (average of exits).

In [ ]:
MRT_GEOJSON_URL = "https://data.gov.sg/api/action/datastore_search?resource_id=d_b39d3a0871985372d7e1637193335da5&limit=5000"

# Try the newer datasets API first, then fall back to direct GeoJSON
MRT_URLS = [
    "https://datamall2.mytransport.sg/ltaodataservice",  # needs key, skip
    "https://data.gov.sg/api/action/datastore_search?resource_id=d_b39d3a0871985372d7e1637193335da5&limit=5000",
]

# Use OneMap theme search for train stations — free, no key required
ONEMAP_SEARCH = "https://www.onemap.gov.sg/api/common/elastic/search"

MRT_STATION_NAMES = [
    "Jurong East MRT", "Bukit Batok MRT", "Bukit Gombak MRT", "Choa Chu Kang MRT",
    "Yew Tee MRT", "Kranji MRT", "Marsiling MRT", "Woodlands MRT",
    "Admiralty MRT", "Sembawang MRT", "Canberra MRT", "Yishun MRT",
    "Khatib MRT", "Ang Mo Kio MRT", "Bishan MRT", "Braddell MRT",
    "Toa Payoh MRT", "Novena MRT", "Newton MRT", "Orchard MRT",
    "Somerset MRT", "Dhoby Ghaut MRT", "City Hall MRT", "Raffles Place MRT",
    "Marina Bay MRT", "Marina South Pier MRT",
    "Pasir Ris MRT", "Tampines MRT", "Simei MRT", "Tanah Merah MRT",
    "Bedok MRT", "Kembangan MRT", "Eunos MRT", "Paya Lebar MRT",
    "Aljunied MRT", "Kallang MRT", "Lavender MRT", "Bugis MRT",
    "Tanjong Pagar MRT", "Outram Park MRT", "Tiong Bahru MRT",
    "Redhill MRT", "Queenstown MRT", "Commonwealth MRT",
    "Buona Vista MRT", "Dover MRT", "Clementi MRT",
    "Chinese Garden MRT", "Lakeside MRT", "Boon Lay MRT",
    "Pioneer MRT", "Joo Koon MRT", "Gul Circle MRT",
    "Tuas Crescent MRT", "Tuas West Road MRT", "Tuas Link MRT",
    "Expo MRT", "Changi Airport MRT",
    "HarbourFront MRT", "Telok Blangah MRT", "Labrador Park MRT",
    "Pasir Panjang MRT", "Haw Par Villa MRT", "Kent Ridge MRT",
    "one-north MRT", "Holland Village MRT", "Farrer Road MRT",
    "Botanic Gardens MRT", "Stevens MRT", "Caldecott MRT",
    "Marymount MRT", "Lorong Chuan MRT", "Serangoon MRT",
    "Bartley MRT", "Tai Seng MRT", "MacPherson MRT",
    "Potong Pasir MRT", "Woodleigh MRT", "Upper Thomson MRT",
    "Lentor MRT", "Mayflower MRT",
    "Bright Hill MRT", "Springleaf MRT",
    "Bayshore MRT", "Bedok North MRT", "Kaki Bukit MRT",
    "Ubi MRT", "Bendemeer MRT", "Geylang Bahru MRT",
    "Mattar MRT", "Dakota MRT", "Mountbatten MRT",
    "Stadium MRT", "Nicoll Highway MRT", "Promenade MRT",
    "Bayfront MRT", "Downtown MRT", "Telok Ayer MRT",
    "Chinatown MRT", "Fort Canning MRT", "Bencoolen MRT",
    "Jalan Besar MRT", "Rochor MRT", "Little India MRT",
    "Farrer Park MRT", "Boon Keng MRT",
    "Hougang MRT", "Buangkok MRT", "Sengkang MRT",
    "Punggol MRT", "Bras Basah MRT", "Esplanade MRT",
    "Tampines West MRT", "Tampines East MRT",
    "Upper Changi MRT", "Pasir Ris East MRT",
    "Woodlands North MRT", "Woodlands South MRT",
    "King Albert Park MRT", "Sixth Avenue MRT", "Tan Kah Kee MRT",
    "Hillview MRT", "Cashew MRT", "Beauty World MRT",
    "Hume MRT",
    "Sungei Kadut MRT",
    "Bukit Panjang LRT", "Senja LRT", "Jelapang LRT",
    "Petir LRT", "Pending LRT", "Bangkit LRT",
    "Fajar LRT", "Bukit Panjang MRT",
    "Sengkang LRT", "Compassvale LRT", "Rumbia LRT",
    "Bakau LRT", "Kangkar LRT", "Ranggung LRT",
    "Cheng Lim LRT", "Farmway LRT", "Kupang LRT",
    "Thanggam LRT", "Fernvale LRT", "Layar LRT",
    "Tongkang LRT", "Renjong LRT",
    "Punggol LRT", "Cove LRT", "Meridian LRT",
    "Coral Edge LRT", "Riviera LRT", "Kadaloor LRT",
    "Oasis LRT", "Damai LRT", "Sam Kee LRT",
    "Teck Lee LRT", "Punggol Point LRT", "Samudera LRT",
    "Nibong LRT", "Sumang LRT", "Soo Teck LRT"
]

rows = []
session = requests.Session()
for name in MRT_STATION_NAMES:
    try:
        r = session.get(ONEMAP_SEARCH, params={
            "searchVal": name,
            "returnGeom": "Y",
            "getAddrDetails": "Y",
            "pageNum": 1
        }, timeout=10)
        data = r.json()
        results = data.get("results", [])
        if results:
            best = results[0]
            lat = float(best.get("LATITUDE", 0))
            lng = float(best.get("LONGITUDE", 0))
            if lat > 0 and lng > 0:
                station_type = "LRT" if "LRT" in name else "MRT"
                rows.append({
                    "name": best.get("SEARCHVAL", name).strip(),
                    "lat": round(lat, 6),
                    "lng": round(lng, 6),
                    "type": station_type,
                    "address": best.get("ADDRESS", "").strip(),
                    "postal_code": best.get("POSTAL", "").strip()
                })
    except Exception as e:
        print(f"  skip {name}: {e}")
    time.sleep(0.15)

mrt_df = pd.DataFrame(rows).drop_duplicates(subset=["name"])
mrt_df.to_csv(OUT_DIR / "mrt_stations.csv", index=False)
print(f"MRT/LRT stations saved: {len(mrt_df)} stations")
mrt_df.head()

## 2. Hawker Centres

Source: NEA Hawker Centres GeoJSON from data.gov.sg.

In [ ]:
HAWKER_GEOJSON_URL = "https://data.gov.sg/api/action/datastore_search?resource_id=d_4a086da0a5553be1d89383cd90d07ecd&limit=5000"

# Try GeoJSON API first
try:
    r = session.get(HAWKER_GEOJSON_URL, timeout=15)
    data = r.json()
    records = data.get("result", {}).get("records", [])
except Exception:
    records = []

if records:
    hawker_rows = []
    for rec in records:
        name = rec.get("name", rec.get("NAME", rec.get("Description", ""))).strip()
        lat = rec.get("latitude", rec.get("LATITUDE", rec.get("lat", 0)))
        lng = rec.get("longitude", rec.get("LONGITUDE", rec.get("lng", 0)))
        addr = rec.get("address_myenv", rec.get("ADDRESS", "")).strip()
        if name and float(lat) > 0 and float(lng) > 0:
            hawker_rows.append({
                "name": name,
                "lat": round(float(lat), 6),
                "lng": round(float(lng), 6),
                "address": addr
            })
    print(f"Hawker centres from data.gov.sg API: {len(hawker_rows)}")
else:
    print("data.gov.sg API returned no records; falling back to OneMap search")
    hawker_rows = []

# Fallback: geocode known hawker centres via OneMap if data.gov API failed
if len(hawker_rows) < 10:
    print("Using OneMap search fallback for hawker centres...")
    r = session.get(ONEMAP_SEARCH, params={
        "searchVal": "hawker centre",
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }, timeout=10)
    total_pages = r.json().get("totalNumPages", 1)
    
    all_results = r.json().get("results", [])
    for page in range(2, min(total_pages + 1, 20)):
        time.sleep(0.2)
        rp = session.get(ONEMAP_SEARCH, params={
            "searchVal": "hawker centre",
            "returnGeom": "Y",
            "getAddrDetails": "Y",
            "pageNum": page
        }, timeout=10)
        all_results.extend(rp.json().get("results", []))

    # Also search for "market" to catch food centres
    for term in ["food centre", "market and food"]:
        rp = session.get(ONEMAP_SEARCH, params={
            "searchVal": term,
            "returnGeom": "Y",
            "getAddrDetails": "Y",
            "pageNum": 1
        }, timeout=10)
        tp = rp.json().get("totalNumPages", 1)
        all_results.extend(rp.json().get("results", []))
        for page in range(2, min(tp + 1, 10)):
            time.sleep(0.2)
            rpp = session.get(ONEMAP_SEARCH, params={
                "searchVal": term,
                "returnGeom": "Y",
                "getAddrDetails": "Y",
                "pageNum": page
            }, timeout=10)
            all_results.extend(rpp.json().get("results", []))

    seen = set()
    hawker_rows = []
    for res in all_results:
        name = res.get("SEARCHVAL", "").strip()
        lat = float(res.get("LATITUDE", 0))
        lng = float(res.get("LONGITUDE", 0))
        key = name.upper()
        if name and lat > 0 and lng > 0 and key not in seen:
            seen.add(key)
            hawker_rows.append({
                "name": name,
                "lat": round(lat, 6),
                "lng": round(lng, 6),
                "address": res.get("ADDRESS", "").strip()
            })

hawker_df = pd.DataFrame(hawker_rows).drop_duplicates(subset=["name"])
hawker_df.to_csv(OUT_DIR / "hawker_centres.csv", index=False)
print(f"Hawker centres saved: {len(hawker_df)}")
hawker_df.head()

## 3. Primary Schools

Source: MOE School Directory from data.gov.sg, geocoded via OneMap Search API.

In [ ]:
SCHOOL_CSV_URL = "https://data.gov.sg/api/action/datastore_search?resource_id=d_688b934f82c1059ed0a6993d2a829089&limit=5000"

try:
    r = session.get(SCHOOL_CSV_URL, timeout=15)
    data = r.json()
    records = data.get("result", {}).get("records", [])
except Exception:
    records = []

school_rows = []
if records:
    primary = [rec for rec in records if "PRIMARY" in str(rec.get("mainlevel_code", "")).upper()]
    print(f"Primary schools from data.gov.sg: {len(primary)}")
    
    for i, rec in enumerate(primary):
        name = rec.get("school_name", "").strip()
        addr = rec.get("address", "").strip()
        postal = str(rec.get("postal_code", "")).strip()
        
        search_q = postal if postal and postal != "nan" else f"{name} SINGAPORE"
        try:
            rg = session.get(ONEMAP_SEARCH, params={
                "searchVal": search_q,
                "returnGeom": "Y",
                "getAddrDetails": "Y",
                "pageNum": 1
            }, timeout=10)
            results = rg.json().get("results", [])
            if results:
                best = results[0]
                lat = float(best.get("LATITUDE", 0))
                lng = float(best.get("LONGITUDE", 0))
                if lat > 0 and lng > 0:
                    school_rows.append({
                        "name": name,
                        "lat": round(lat, 6),
                        "lng": round(lng, 6),
                        "address": addr,
                        "postal_code": postal,
                        "type": "PRIMARY"
                    })
        except Exception as e:
            print(f"  skip {name}: {e}")
        
        if (i + 1) % 20 == 0:
            print(f"  geocoded {i+1}/{len(primary)}")
        time.sleep(0.15)
else:
    print("data.gov.sg school API returned no records; using OneMap search fallback")
    r = session.get(ONEMAP_SEARCH, params={
        "searchVal": "primary school",
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }, timeout=10)
    total_pages = r.json().get("totalNumPages", 1)
    all_results = r.json().get("results", [])
    for page in range(2, min(total_pages + 1, 30)):
        time.sleep(0.2)
        rp = session.get(ONEMAP_SEARCH, params={
            "searchVal": "primary school",
            "returnGeom": "Y",
            "getAddrDetails": "Y",
            "pageNum": page
        }, timeout=10)
        all_results.extend(rp.json().get("results", []))
    
    seen = set()
    for res in all_results:
        name = res.get("SEARCHVAL", "").strip()
        if "PRIMARY" not in name.upper() and "SCHOOL" not in name.upper():
            continue
        lat = float(res.get("LATITUDE", 0))
        lng = float(res.get("LONGITUDE", 0))
        key = name.upper()
        if name and lat > 0 and lng > 0 and key not in seen:
            seen.add(key)
            school_rows.append({
                "name": name,
                "lat": round(lat, 6),
                "lng": round(lng, 6),
                "address": res.get("ADDRESS", "").strip(),
                "postal_code": res.get("POSTAL", "").strip(),
                "type": "PRIMARY"
            })

school_df = pd.DataFrame(school_rows).drop_duplicates(subset=["name"])
school_df.to_csv(OUT_DIR / "schools.csv", index=False)
print(f"Schools saved: {len(school_df)}")
school_df.head()

## 4. Shopping Malls

Source: OneMap POI search for "shopping mall" and "shopping centre".

In [ ]:
all_mall_results = []
for term in ["shopping mall", "shopping centre", "shopping center"]:
    r = session.get(ONEMAP_SEARCH, params={
        "searchVal": term,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }, timeout=10)
    resp = r.json()
    total_pages = resp.get("totalNumPages", 1)
    all_mall_results.extend(resp.get("results", []))
    for page in range(2, min(total_pages + 1, 15)):
        time.sleep(0.2)
        rp = session.get(ONEMAP_SEARCH, params={
            "searchVal": term,
            "returnGeom": "Y",
            "getAddrDetails": "Y",
            "pageNum": page
        }, timeout=10)
        all_mall_results.extend(rp.json().get("results", []))

seen = set()
mall_rows = []
for res in all_mall_results:
    name = res.get("SEARCHVAL", "").strip()
    lat = float(res.get("LATITUDE", 0))
    lng = float(res.get("LONGITUDE", 0))
    key = name.upper()
    if name and lat > 0 and lng > 0 and key not in seen:
        seen.add(key)
        mall_rows.append({
            "name": name,
            "lat": round(lat, 6),
            "lng": round(lng, 6),
            "address": res.get("ADDRESS", "").strip()
        })

mall_df = pd.DataFrame(mall_rows).drop_duplicates(subset=["name"])
mall_df.to_csv(OUT_DIR / "malls.csv", index=False)
print(f"Malls saved: {len(mall_df)}")
mall_df.head()

## Summary

In [ ]:
print("\n=== Amenity Data Summary ===")
for f in sorted(OUT_DIR.glob("*.csv")):
    df = pd.read_csv(f)
    print(f"  {f.name:25s}  {len(df):>4d} records  cols: {list(df.columns)}")
print(f"\nAll files saved to: {OUT_DIR}")